# Set this microscope's limits

The ZEN API does not say how far the stage and the focus may travel, so you show the driver. The limits are saved in this microscope's settings and every later connection obeys them: a move outside them is refused before anything is sent.

All numbers are micrometres in ZEN's own coordinates, as ZEN's stage and focus displays show them.

## Start from the current limits

`objective_position`, `experiment` and `set_z_stack` are `[]`: reviewed, no limit. Write `{"allowed": [...]}` to fence the driver to some objective positions (counted from 1) or some experiments.

In [ ]:
import bootstrap

from zmart_drivers.zeiss.light_microscope.zen_api_grpc.driver_setup.limits.config import (
    adopt_limits,
    validate_limits,
)

LIMITS = {
    "x_um": {"range": [0, 100000]},
    "y_um": {"range": [0, 75000]},
    "z_um": {"range": [0, 10000]},
    "objective_position": [],
    "experiment": [],
    "set_z_stack": [],
}

## Measure

Connect, then drive the stage with the joystick or in ZEN to each safe corner of its travel and run that corner's cell. Run a cell again after moving the stage to record that corner again.

In [ ]:
import zmart_drivers.zeiss.light_microscope.zen_api_grpc as zen

# The gateway's control token: copy it from the ZEN API Gateway's tray icon, or leave it
# out when this computer runs the gateway (the driver reads its token file then).
scope = zen.connect(password=None, experiment=None)

from zmart_drivers.zeiss.light_microscope.zen_api_grpc.driver_setup.limits.measure_limits import (
    CornerRecorder,
)

corners = CornerRecorder(scope.client)

**Corner 1:** drive the stage to the first safe corner, then run the cell below.

In [ ]:
corners.record(1)

**Corner 2:** the second safe corner.

In [ ]:
corners.record(2)

**Corner 3:** the third safe corner.

In [ ]:
corners.record(3)

**Corner 4:** the fourth safe corner.

In [ ]:
corners.record(4)

**The focus (optional):** lower the objective to its lowest safe position and run the first cell, then raise it to its highest safe position, clear of the sample, and run the second. Leave both out to keep the focus range written above.

In [ ]:
corners.record_focus("low")

In [ ]:
corners.record_focus("high")

In [ ]:
measured = corners.limits()
LIMITS.update(measured["limits"])
print("x:", LIMITS["x_um"]["range"])
print("y:", LIMITS["y_um"]["range"])
print("z:", LIMITS["z_um"]["range"])

## Check

In [ ]:
validated_limits = validate_limits(LIMITS)

## Save and adopt

This saves the notebook, checks that the measurement and the check ran, and publishes the limits together with the notebook, which records the corners.

In [ ]:
published = bootstrap.NOTEBOOK.save_and_adopt(
    lambda notebook: adopt_limits(validated_limits, notebook_paths=[notebook])
)
scope.close()
published